# Práctica 4: Análisis de Logs de Seguridad (SIEM)

TODO: Agregar contexto y portada

## Fase 1 - Diagnóstico del Dataset

### 1.1 Carga del dataset y tipos
Carga el dataset. Muestra `shape`, `dtypes` e `info()`. ¿Cuántas columnas tienen tipo incorrecto?

**Apartado de código:** usar la celda siguiente.

In [2]:
from pathlib import Path

import pandas as pd

import utils

DATASET_PATH = Path("content/siem_eventos_sucio - siem_eventos_sucio.csv")

df_raw = utils.load_siem_dataset(DATASET_PATH)
df = df_raw.copy()

display(df.head())
utils.styled_output(str(df.shape), "Shape del dataset")
utils.styled_output(df.dtypes.astype(str).to_string(), "dtypes actuales")
utils.styled_output(utils.dataframe_info_text(df), "info()")

expected_dtypes = {
    "timestamp_evento": "datetime64[ns]",
    "timestamp_resolucion": "datetime64[ns]",
    "puerto_destino": "Int64",
}
incorrect_type_report = [
    f"- {column}: esperado {expected_dtype}, actual {df[column].dtype}"
    for column, expected_dtype in expected_dtypes.items()
]
utils.styled_output(
    "\n".join(incorrect_type_report),
    f"Columnas con tipo incorrecto: {len(incorrect_type_report)}",
)

,evento_id,timestamp_evento,timestamp_resolucion,ip_origen,ip_destino,puerto_destino,protocolo,tipo_evento,categoria,severidad,...,sistema_operativo,pais_origen,usuario,analista_id,bytes_enviados,bytes_recibidos,accion_tomada,tiempo_respuesta_min,falso_positivo,resuelto
0,EVT101614,2024-08-23 21:11:26,2024-08-23 22:07:26,151.70.241.104,10.9.34.76,3306.0,FTP,Phishing detectado,Ingeniería social,Media,...,Windows 11 Pro,México,user_martinez11,ANA008,1036785,280295,Cuarentenado,56,False,True
1,EVT104241,03/16/2024 13:18,2024-03-16 11:53:00,66.238.112.228,10.8.47.206,8443.0,SMB,Exfiltración de datos,Exfiltración,Crítica,...,Linux Debian 11,Irán,user_garcia88,ANA009,1918856,949645,Cuarentenado,50,False,True
2,EVT104669,2024-10-30 5:31:03,NaN,213.2.168.63,10.3.13.216,3389.0,RDP,Ransomware detectado,Malware,Crítica,...,Linux Ubuntu 22.04,China,user_gonzalez57,ANA010,117858,1230502,Alertado,28,False,True
3,EVT102694,2024-02-23 13:05:27,2024-02-23 15:13:27,167.175.67.104,10.0.43.59,80.0,TCP,Escalación de privilegios,Control de acceso,Alta,...,Cisco IOS,Brasil,NaN,ANA008,3326046,1756088,Bloqueado,128,False,True
4,EVT104256,2024-05-28 7:24:59,2024-05-28 9:58:59,45.60.51.96,10.2.43.122,3306.0,HTTPS,Acceso no autorizado,Control de acceso,Alta,...,Windows Server 2022,Estados Unidos,NaN,ANA010,1309398,1415415,Bloqueado,154,False,True


# Shape del dataset
(5515, 21)
# dtypes actuales
evento_id                   str
timestamp_evento            str
timestamp_resolucion        str
ip_origen                   str
ip_destino                  str
puerto_destino          float64
protocolo                   str
tipo_evento                 str
categoria                   str
severidad                   str
sistema_afectado            str
sistema_operativo           str
pais_origen                 str
usuario                     str
analista_id                 str
bytes_enviados            int64
bytes_recibidos           int64
accion_tomada               str
tiempo_respuesta_min      int64
falso_positivo             bool
resuelto                   bool
# info()
<class 'pandas.DataFrame'>
RangeIndex: 5515 entries, 0 to 5514
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   evento_id             5515 non-null   str    
 1   timestamp_evento

### 1.2 Nulos por columna
Calcula el porcentaje de nulos por columna. Identifica cuáles nulos son esperados y lógicos.

**Apartado de código:** usar la celda siguiente.

In [3]:
null_summary = utils.null_percentage_summary(df)
display(null_summary.query("nulos > 0"))

expected_nulls = [
    "- timestamp_resolucion: es normal cuando el evento sigue abierto.",
    "- usuario: muchos eventos de red no están asociados a una cuenta concreta.",
]
unexpected_nulls = [
    "- analista_id: deja eventos sin responsable asignado.",
    "- pais_origen: reduce la calidad del análisis geográfico.",
    "- puerto_destino: impide validar con precisión el servicio objetivo.",
    "- sistema_operativo: complica el análisis por plataforma.",
]
utils.styled_output(
    "\n".join(expected_nulls + [""] + unexpected_nulls),
    "Interpretación de nulos",
)

,nulos,porcentaje
timestamp_resolucion,1021,18.51
usuario,864,15.67
analista_id,20,0.36
pais_origen,15,0.27
puerto_destino,12,0.22
sistema_operativo,10,0.18


# Interpretación de nulos
- timestamp_resolucion: es normal cuando el evento sigue abierto.
- usuario: muchos eventos de red no están asociados a una cuenta concreta.

- analista_id: deja eventos sin responsable asignado.
- pais_origen: reduce la calidad del análisis geográfico.
- puerto_destino: impide validar con precisión el servicio objetivo.
- sistema_operativo: complica el análisis por plataforma.


### 1.3 Valores únicos en columnas categóricas
Muestra valores únicos de: `tipo_evento`, `severidad`, `accion_tomada`, `categoria`, `pais_origen`.

**Apartado de código:** usar la celda siguiente.

In [4]:
categorical_columns = [
    "tipo_evento",
    "severidad",
    "accion_tomada",
    "categoria",
    "pais_origen",
]

for column in categorical_columns:
    utils.styled_output(column, "Columna analizada")
    display(utils.top_value_counts(df, column, dropna=False, top_n=20))

# Columna analizada
tipo_evento


,conteo
tipo_evento,
Ransomware detectado,381
Escalación de privilegios,381
Exfiltración de datos,378
XSS detectado,372
Fuerza bruta,366
Escaneo de red,360
DDoS,358
Phishing detectado,356
Login exitoso,352


# Columna analizada
severidad


,conteo
severidad,
Crítica,1661
Alta,1509
Media,1396
Baja,811
ALTA,26
CRÍTICA,23
crítica,16
alta,13
critica,12


# Columna analizada
accion_tomada


,conteo
accion_tomada,
Alertado,1482
Bloqueado,1264
Cuarentenado,1162
En revisión,1146
Permitido,351
BLOQUEADO,21
bloqueado,17
EN REVISIÓN,12
CUARENTENADO,11


# Columna analizada
categoria


,conteo
categoria,
Autenticación,1063
Control de acceso,731
Web,723
Malware,720
Reconocimiento,714
Exfiltración,381
Disponibilidad,365
Ingeniería social,363
Post-explotación,355


# Columna analizada
pais_origen


,conteo
pais_origen,
México,1743
India,356
Irán,355
Holanda,354
Ucrania,351
Corea del Norte,350
Rusia,336
China,327
Alemania,321


### 1.4 Validación visual de `ip_origen`
Detecta visualmente IPs con formato inválido de IPv4.

**Apartado de código:** usar la celda siguiente.

In [5]:
invalid_ip_examples = utils.invalid_ipv4_examples(df["ip_origen"])
utils.styled_output(
    "\n".join(f"- {ip}" for ip in invalid_ip_examples[:10]),
    f"Ejemplos de IPs inválidas detectadas: {len(invalid_ip_examples)}",
)

display(
    df.loc[df["ip_origen"].astype(str).isin(invalid_ip_examples), ["evento_id", "ip_origen"]]
    .drop_duplicates()
    .head(10)
)

# Ejemplos de IPs inválidas detectadas: 5
- 192.168.1
- 10.0.0.999
- 256.1.1.1
- 10.0..5
- abc.def.ghi.jkl


,evento_id,ip_origen
410,EVT103285,192.168.1
1013,EVT101126,10.0.0.999
1723,EVT101162,192.168.1
2047,EVT101697,256.1.1.1
2491,EVT101659,10.0..5
2511,EVT100839,10.0.0.999
2552,EVT101089,abc.def.ghi.jkl
2557,EVT100930,192.168.1
2747,EVT103548,10.0.0.999
3736,EVT102003,192.168.1


### 1.5 Estadística descriptiva de numéricas
Aplica `describe()` a columnas numéricas e identifica columnas con valores físicamente imposibles (negativos, etc.).

**Apartado de código:** usar la celda siguiente.

In [6]:
numeric_columns = [
    "puerto_destino",
    "bytes_enviados",
    "bytes_recibidos",
    "tiempo_respuesta_min",
]

display(df[numeric_columns].describe().T)

impossible_values_summary = pd.DataFrame(
    {
        "regla": [
            "puerto_destino <= 0",
            "puerto_destino > 65535",
            "bytes_enviados < 0",
            "bytes_recibidos < 0",
            "tiempo_respuesta_min <= 0",
        ],
        "casos": [
            int((pd.to_numeric(df["puerto_destino"], errors="coerce") <= 0).sum()),
            int((pd.to_numeric(df["puerto_destino"], errors="coerce") > 65535).sum()),
            int((pd.to_numeric(df["bytes_enviados"], errors="coerce") < 0).sum()),
            int((pd.to_numeric(df["bytes_recibidos"], errors="coerce") < 0).sum()),
            int((pd.to_numeric(df["tiempo_respuesta_min"], errors="coerce") <= 0).sum()),
        ],
    }
)
display(impossible_values_summary)
utils.styled_output(
    "Las columnas que no deberían tener valores negativos son puerto_destino, bytes_enviados, bytes_recibidos y tiempo_respuesta_min.",
    "Lectura del describe()",
)

,count,mean,std,min,25%,50%,75%,max
puerto_destino,5503.0,2.293848e+03,4.637248e+03,-1.0,25.0,443.0,3389.0,100000.0
bytes_enviados,5515.0,2.566948e+06,2.134233e+06,-500.0,1219062.5,2495767.0,3743576.5,40809824.0
bytes_recibidos,5515.0,1.004274e+06,5.761712e+05,458.0,512422.5,991216.0,1507670.0,1999871.0
tiempo_respuesta_min,5515.0,2.992718e+02,7.574097e+02,-10.0,39.0,103.0,284.0,9993.0


,regla,casos
0,puerto_destino <= 0,2
1,puerto_destino > 65535,8
2,bytes_enviados < 0,6
3,bytes_recibidos < 0,0
4,tiempo_respuesta_min <= 0,6


# Lectura del describe()
Las columnas que no deberían tener valores negativos son puerto_destino, bytes_enviados, bytes_recibidos y tiempo_respuesta_min.


### 1.6 Lista de problemas detectados

- Hay 3 columnas con tipo claramente incorrecto para el análisis: `timestamp_evento`, `timestamp_resolucion` y `puerto_destino`.
- Existen nulos lógicos en `timestamp_resolucion` y `usuario`, porque un evento puede seguir abierto o no estar asociado a una cuenta.
- También hay nulos no ideales en `analista_id`, `pais_origen`, `puerto_destino` y `sistema_operativo`.
- Las columnas categóricas tienen ruido textual: mayúsculas inconsistentes, espacios y variantes duplicadas del mismo valor.
- `severidad` contiene sinónimos no estándar como `Grave` y `Urgente`.
- `accion_tomada` incluye valores fuera del catálogo esperado, por ejemplo `Ignorado` y `Cuarentena`.
- `pais_origen` mezcla sinónimos para el mismo país, como `Russia`/`Rusia` y `EEUU`/`United States`/`Estados Unidos`.
- `ip_origen` contiene direcciones con formato IPv4 inválido, por ejemplo `192.168.1`, `abc.def.ghi.jkl`, `10.0.0.999` y `256.1.1.1`.
- En variables numéricas hay valores físicamente imposibles: puertos fuera de rango, `bytes_enviados` negativos y `tiempo_respuesta_min` menor o igual a cero.
- El dataset contiene 15 duplicados exactos que deben eliminarse en la limpieza.

## Fase 2 - Limpieza de Texto y Estandarización

### 2.1 Normalización de texto
Aplica normalización (`strip()`, `title()`) a: `tipo_evento`, `severidad`, `accion_tomada`, `categoria`, `pais_origen`, `protocolo`, `sistema_operativo`.

**Apartado de código:** usar la celda siguiente.

In [7]:
# Reiniciamos desde el dataset original para separar diagnóstico y limpieza.
df = df_raw.copy()

text_columns = [
    "tipo_evento",
    "severidad",
    "accion_tomada",
    "categoria",
    "pais_origen",
    "protocolo",
    "sistema_operativo",
]

df = utils.normalize_text_columns(df, text_columns)

# Correcciones puntuales para etiquetas técnicas que title() no deja en el formato deseado.
df["tipo_evento"] = df["tipo_evento"].replace(
    {
        "Ddos": "DDoS",
        "Xss Detectado": "XSS Detectado",
        "Inyección Sql": "Inyección SQL",
    }
)
df["protocolo"] = df["protocolo"].str.upper()

display(df[text_columns].head())

,tipo_evento,severidad,accion_tomada,categoria,pais_origen,protocolo,sistema_operativo
0,Phishing Detectado,Media,Cuarentenado,Ingeniería Social,México,FTP,Windows 11 Pro
1,Exfiltración De Datos,Crítica,Cuarentenado,Exfiltración,Irán,SMB,Linux Debian 11
2,Ransomware Detectado,Crítica,Alertado,Malware,China,RDP,Linux Ubuntu 22.04
3,Escalación De Privilegios,Alta,Bloqueado,Control De Acceso,Brasil,TCP,Cisco Ios
4,Acceso No Autorizado,Alta,Bloqueado,Control De Acceso,Estados Unidos,HTTPS,Windows Server 2022


### 2.2 Estandarización de severidad
Deja solo: `Crítica`, `Alta`, `Media`, `Baja`. Define y documenta mapeo de `Grave` y `Urgente`.

**Apartado de código:** usar la celda siguiente.

In [8]:
# Decisión de negocio: 'Grave' y 'Urgente' se homologan a 'Alta'
# porque expresan riesgo elevado, pero no forman parte del catálogo oficial del SOC.
severity_map = {
    "Critica": "Crítica",
    "Crítica": "Crítica",
    "Cr�tica": "Crítica",
    "Grave": "Alta",
    "Urgente": "Alta",
}
df["severidad"] = df["severidad"].replace(severity_map)

display(utils.top_value_counts(df, "severidad", dropna=False))

unexpected_severity = sorted(set(df["severidad"].dropna()) - {"Crítica", "Alta", "Media", "Baja"})
utils.styled_output(
    "Valores fuera del catálogo: " + (", ".join(unexpected_severity) if unexpected_severity else "ninguno"),
    "Validación de severidad",
)

,conteo
severidad,
Crítica,1712
Alta,1561
Media,1418
Baja,824


# Validación de severidad
Valores fuera del catálogo: ninguno


### 2.3 Estandarización de `accion_tomada`
Catálogo objetivo: `Bloqueado`, `Permitido`, `En Revisión`, `Alertado`, `Cuarentenado`. Define qué hacer con `Ignorado`.

**Apartado de código:** usar la celda siguiente.

In [9]:
# Decisión de negocio: 'Ignorado' se homologa a 'Permitido'
# porque implica que no hubo contención real sobre el evento.
action_map = {
    "En Revision": "En Revisión",
    "En Revisi�n": "En Revisión",
    "Cuarentena": "Cuarentenado",
    "Ignorado": "Permitido",
}
df["accion_tomada"] = df["accion_tomada"].replace(action_map)

display(utils.top_value_counts(df, "accion_tomada", dropna=False))

expected_actions = {"Bloqueado", "Permitido", "En Revisión", "Alertado", "Cuarentenado"}
unexpected_actions = sorted(set(df["accion_tomada"].dropna()) - expected_actions)
utils.styled_output(
    "Valores fuera del catálogo: " + (", ".join(unexpected_actions) if unexpected_actions else "ninguno"),
    "Validación de accion_tomada",
)

,conteo
accion_tomada,
Alertado,1497
Bloqueado,1302
Cuarentenado,1189
En Revisión,1165
Permitido,362


# Validación de accion_tomada
Valores fuera del catálogo: ninguno


### 2.4 Unificación de `pais_origen`
Unifica sinónimos: `Russia`/`Rusia`; `EEUU`/`United States`/`Estados Unidos`.

**Apartado de código:** usar la celda siguiente.

In [10]:
country_map = {
    "Russia": "Rusia",
    "Eeuu": "Estados Unidos",
    "United States": "Estados Unidos",
    "M�xico": "México",
    "Ir�n": "Irán",
    "Corea Del Norte": "Corea del Norte",
}
df["pais_origen"] = df["pais_origen"].replace(country_map)

display(utils.top_value_counts(df, "pais_origen", dropna=False, top_n=15))

,conteo
pais_origen,
México,1758
India,362
Irán,359
Holanda,356
Ucrania,356
Corea del Norte,352
Rusia,349
China,335
Estados Unidos,333


### 2.5 Duplicados exactos
Detecta y elimina duplicados exactos. Reporta cuántos se eliminaron.

**Apartado de código:** usar la celda siguiente.

In [11]:
duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

utils.styled_output(
    f"Duplicados exactos detectados: {duplicates_before}\nFilas después de eliminarlos: {len(df)}",
    "Resultado de deduplicación",
)

# Resultado de deduplicación
Duplicados exactos detectados: 15
Filas después de eliminarlos: 5500


## Fase 3 - Timestamps y Tipos de Datos

### 3.1 Conversión de `timestamp_evento`
Convierte a datetime considerando mezcla de formatos (incluye MM/DD/YYYY).

**Apartado de código:** usar la celda siguiente.

In [12]:
import importlib

importlib.reload(utils)

timestamp_evento_original = df["timestamp_evento"].copy()
timestamp_evento_convertido = utils.parse_mixed_datetime(timestamp_evento_original, dayfirst=False)
timestamp_evento_fallidos = utils.datetime_parse_failures(
    timestamp_evento_original,
    timestamp_evento_convertido,
)

df["timestamp_evento"] = timestamp_evento_convertido

utils.styled_output(
    (
        f"Registros analizados: {len(df)}\n"
        f"Fechas convertidas correctamente: {int(df['timestamp_evento'].notna().sum())}\n"
        f"Valores no parseables: {int(timestamp_evento_fallidos.shape[0])}"
    ),
    "Conversión de timestamp_evento",
)

if timestamp_evento_fallidos.empty:
    utils.styled_output(
        "Todos los valores de timestamp_evento se pudieron convertir con formato mixto.",
        "Resultado 3.1",
    )
else:
    display(timestamp_evento_fallidos.astype(str).value_counts().rename("conteo").to_frame())

display(df[["timestamp_evento"]].head())

# Conversión de timestamp_evento
Registros analizados: 5500
Fechas convertidas correctamente: 5500
Valores no parseables: 0
# Resultado 3.1
Todos los valores de timestamp_evento se pudieron convertir con formato mixto.


,timestamp_evento
0,2024-08-23 21:11:26
1,2024-03-16 13:18:00
2,2024-10-30 05:31:03
3,2024-02-23 13:05:27
4,2024-05-28 07:24:59


### 3.2 Conversión de `timestamp_resolucion`
Convierte a datetime y explica por qué no usar `errors='coerce'` sin análisis previo.

**Apartado de código y justificación:** usar celdas siguientes.

In [13]:
timestamp_resolucion_original = df["timestamp_resolucion"].copy()
timestamp_resolucion_convertido = utils.parse_mixed_datetime(timestamp_resolucion_original, dayfirst=False)
timestamp_resolucion_fallidos = utils.datetime_parse_failures(
    timestamp_resolucion_original,
    timestamp_resolucion_convertido,
)

nulos_legitimos_resolucion = int(timestamp_resolucion_original.isna().sum())
df["timestamp_resolucion"] = timestamp_resolucion_convertido

utils.styled_output(
    (
        f"Nulos originales preservados: {nulos_legitimos_resolucion}\n"
        f"Fechas convertidas correctamente: {int(df['timestamp_resolucion'].notna().sum())}\n"
        f"Valores no parseables distintos de null: {int(timestamp_resolucion_fallidos.shape[0])}"
    ),
    "Conversión de timestamp_resolucion",
)

if timestamp_resolucion_fallidos.empty:
    utils.styled_output(
        "Los null observados son consistentes con eventos aún no resueltos; no se detectaron strings inválidos de fecha.",
        "Resultado 3.2",
    )
else:
    display(timestamp_resolucion_fallidos.astype(str).value_counts().rename("conteo").to_frame())

display(df[["timestamp_evento", "timestamp_resolucion", "resuelto"]].head())

# Conversión de timestamp_resolucion
Nulos originales preservados: 1017
Fechas convertidas correctamente: 4483
Valores no parseables distintos de null: 0
# Resultado 3.2
Los null observados son consistentes con eventos aún no resueltos; no se detectaron strings inválidos de fecha.


,timestamp_evento,timestamp_resolucion,resuelto
0,2024-08-23 21:11:26,2024-08-23 22:07:26,True
1,2024-03-16 13:18:00,2024-03-16 11:53:00,True
2,2024-10-30 05:31:03,NaT,True
3,2024-02-23 13:05:27,2024-02-23 15:13:27,True
4,2024-05-28 07:24:59,2024-05-28 09:58:59,True


**Justificación 3.2:** no conviene usar `errors='coerce'` a ciegas porque convertiría cualquier string mal formado en `NaT` y mezclaría dos casos muy distintos: nulos legítimos de eventos no resueltos y errores reales de calidad de datos. Primero conviene contar los null originales y comparar contra los valores que fallan al parsear; así conservamos los null esperados y detectamos si existen fechas corruptas que merecen revisión.

### 3.3 Conversión de columnas numéricas
Convierte `bytes_enviados`, `bytes_recibidos`, `puerto_destino`, `tiempo_respuesta_min` a enteros, manejando nulos con `Int64`.

**Apartado de código:** usar la celda siguiente.

In [14]:
numeric_columns_phase3 = [
    "bytes_enviados",
    "bytes_recibidos",
    "puerto_destino",
    "tiempo_respuesta_min",
]

df = utils.convert_nullable_integer_columns(df, numeric_columns_phase3)

dtypes_numericos = df[numeric_columns_phase3].dtypes.astype(str).rename("dtype").to_frame()
nulos_numericos = df[numeric_columns_phase3].isna().sum().rename("nulos").to_frame()

display(dtypes_numericos)
display(nulos_numericos)
utils.styled_output(
    "Las cuatro columnas ya usan Int64, lo que permite conservar nulos sin perder comportamiento de enteros.",
    "Resultado 3.3",
)

,dtype
bytes_enviados,Int64
bytes_recibidos,Int64
puerto_destino,Int64
tiempo_respuesta_min,Int64


,nulos
bytes_enviados,0
bytes_recibidos,0
puerto_destino,12
tiempo_respuesta_min,0


# Resultado 3.3
Las cuatro columnas ya usan Int64, lo que permite conservar nulos sin perder comportamiento de enteros.


### 3.4 Conversión de booleanos
Convierte `falso_positivo` y `resuelto` a tipo booleano.

**Apartado de código:** usar la celda siguiente.

In [15]:
boolean_columns = ["falso_positivo", "resuelto"]

df = utils.convert_boolean_columns(df, boolean_columns)

display(df[boolean_columns].dtypes.astype(str).rename("dtype").to_frame())
for column in boolean_columns:
    display(df[column].value_counts(dropna=False).rename(column).to_frame())

utils.styled_output(
    "Se convirtieron a booleano nullable para mantener consistencia y tolerar posibles nulos futuros.",
    "Resultado 3.4",
)

,dtype
falso_positivo,boolean
resuelto,boolean


,falso_positivo
falso_positivo,
False,4730
True,770


,resuelto
resuelto,
True,4509
False,991


# Resultado 3.4
Se convirtieron a booleano nullable para mantener consistencia y tolerar posibles nulos futuros.


## Fase 4 - Valores Imposibles y Outliers

### 4.1 Validación de puertos
Detecta y elimina puertos fuera del rango 1-65535.

**Apartado de código:** usar la celda siguiente.

In [16]:
invalid_port_mask = df["puerto_destino"].notna() & ~df["puerto_destino"].between(1, 65535)
invalid_port_rows = df.loc[
    invalid_port_mask,
    ["evento_id", "puerto_destino", "tipo_evento", "sistema_afectado"],
].copy()

utils.styled_output(
    (
        f"Puertos inválidos detectados: {int(invalid_port_mask.sum())}\n"
        f"Puertos nulos conservados: {int(df['puerto_destino'].isna().sum())}"
    ),
    "Validación de puertos",
)

display(invalid_port_rows.head(10))

df = df.loc[~invalid_port_mask].copy()
utils.styled_output(
    f"Filas restantes después de eliminar puertos fuera del rango 1-65535: {len(df)}",
    "Resultado 4.1",
)

# Validación de puertos
Puertos inválidos detectados: 10
Puertos nulos conservados: 12


,evento_id,puerto_destino,tipo_evento,sistema_afectado
907,EVT104181,65536,Login Exitoso,NAS-BACKUP-01
1168,EVT100566,100000,Fuerza Bruta,SRV-AD-REPLICA
1876,EVT102980,100000,Ransomware Detectado,APP-ERP-PROD
2301,EVT100222,-1,Login Fallido,SRV-DB-DEV
2950,EVT102460,-1,Malware Detectado,SRV-MAIL-01
3069,EVT104702,100000,Movimiento Lateral,WS-RH-005
4254,EVT104401,99999,Escaneo De Red,SRV-AD-REPLICA
4331,EVT104197,99999,DDoS,SRV-DB-DEV
4943,EVT104268,99999,Escalación De Privilegios,WS-FIN-022
5346,EVT103743,100000,Movimiento Lateral,SRV-AD-PRIMARY


# Resultado 4.1
Filas restantes después de eliminar puertos fuera del rango 1-65535: 5490


### 4.2 Valores negativos o cero
Detecta filas con `bytes_enviados` o `tiempo_respuesta_min` negativos o cero y justifica la decisión de tratamiento.

**Apartado de código y justificación:** usar celdas siguientes.

In [17]:
bytes_no_positivos = df["bytes_enviados"].le(0).fillna(False)
tiempo_no_positivo = df["tiempo_respuesta_min"].le(0).fillna(False)
valores_no_positivos_mask = bytes_no_positivos | tiempo_no_positivo

resumen_no_positivos = pd.DataFrame(
    {
        "casos": [
            int(bytes_no_positivos.sum()),
            int(tiempo_no_positivo.sum()),
            int(valores_no_positivos_mask.sum()),
        ]
    },
    index=[
        "bytes_enviados <= 0",
        "tiempo_respuesta_min <= 0",
        "filas a eliminar",
    ],
)

display(resumen_no_positivos)
display(
    df.loc[
        valores_no_positivos_mask,
        ["evento_id", "bytes_enviados", "tiempo_respuesta_min", "tipo_evento", "resuelto"],
    ].head(10)
)

df = df.loc[~valores_no_positivos_mask].copy()
utils.styled_output(
    f"Filas restantes después de eliminar registros con bytes_enviados o tiempo_respuesta_min no positivos: {len(df)}",
    "Resultado 4.2",
)

,casos
bytes_enviados <= 0,8
tiempo_respuesta_min <= 0,5
filas a eliminar,13


,evento_id,bytes_enviados,tiempo_respuesta_min,tipo_evento,resuelto
950,EVT105061,-500,34,Inyección SQL,True
1902,EVT101401,0,30,Login Fallido,True
2761,EVT105361,0,30,Exfiltración De Datos,True
3326,EVT100363,-500,357,XSS Detectado,True
3487,EVT100035,753045,-1,Ransomware Detectado,True
3640,EVT100986,-500,582,Login Exitoso,False
3751,EVT101628,-500,9,Fuerza Bruta,True
3761,EVT100300,1584488,-10,Fuerza Bruta,True
3908,EVT102054,4319268,-1,Acceso No Autorizado,True
4242,EVT100963,-1,526,Login Exitoso,True


# Resultado 4.2
Filas restantes después de eliminar registros con bytes_enviados o tiempo_respuesta_min no positivos: 5477


**Justificación 4.2:** para este caso conviene eliminar estas filas. `bytes_enviados <= 0` contradice la idea de tráfico registrado desde el origen del evento, y `tiempo_respuesta_min <= 0` rompe la lógica física del proceso de atención. Imputarlos introduciría valores inventados en métricas operativas; por eso es más sólido excluirlos del dataset limpio.

### 4.3 Validación de IPv4
Valida `ip_origen` (4 octetos, cada uno entre 0 y 255) y detecta IPs inválidas con Python.

**Apartado de código:** usar la celda siguiente.

In [18]:
invalid_ip_mask = utils.invalid_ipv4_mask(df["ip_origen"])
invalid_ip_examples_phase4 = utils.invalid_ipv4_examples(df["ip_origen"])
invalid_ip_rows = df.loc[
    invalid_ip_mask,
    ["evento_id", "ip_origen", "ip_destino", "tipo_evento"],
].copy()

utils.styled_output(
    (
        f"Filas con IP origen inválida: {int(invalid_ip_mask.sum())}\n"
        "Ejemplos: " + ", ".join(invalid_ip_examples_phase4[:5])
    ),
    "Validación de IPv4",
)

display(invalid_ip_rows.head(10))

df = df.loc[~invalid_ip_mask].copy()
utils.styled_output(
    f"Filas restantes después de eliminar IPs inválidas: {len(df)}",
    "Resultado 4.3",
)

# Validación de IPv4
Filas con IP origen inválida: 17
Ejemplos: 192.168.1, 10.0.0.999, 256.1.1.1, 10.0..5, abc.def.ghi.jkl


,evento_id,ip_origen,ip_destino,tipo_evento
410,EVT103285,192.168.1,10.10.47.148,Escalación De Privilegios
1013,EVT101126,10.0.0.999,10.2.4.113,Phishing Detectado
1723,EVT101162,192.168.1,10.1.37.28,Ransomware Detectado
2047,EVT101697,256.1.1.1,10.7.50.216,Phishing Detectado
2491,EVT101659,10.0..5,10.1.50.78,Movimiento Lateral
2511,EVT100839,10.0.0.999,10.9.13.130,Exfiltración De Datos
2552,EVT101089,abc.def.ghi.jkl,10.4.8.158,Acceso No Autorizado
2557,EVT100930,192.168.1,10.7.0.162,Login Exitoso
2747,EVT103548,10.0.0.999,10.8.32.96,Escaneo De Puertos
3736,EVT102003,192.168.1,10.10.6.25,Escaneo De Puertos


# Resultado 4.3
Filas restantes después de eliminar IPs inválidas: 5460


## Fase 5 - Inconsistencias Lógicas (Nivel SOC)

### 5.1 Falso positivo bloqueado
Detecta casos donde `falso_positivo = True` y `accion_tomada` sea `Bloqueado` o `Cuarentenado`. Decide qué campo corregir.

**Apartado de código:** usar la celda siguiente.

In [19]:
import importlib

importlib.reload(utils)

mask_fp_agresivo = utils.false_positive_aggressive_action_mask(df)
fp_agresivo_rows = df.loc[
    mask_fp_agresivo,
    ["evento_id", "tipo_evento", "accion_tomada", "falso_positivo", "severidad"],
] .copy()

utils.styled_output(
    (
        f"Casos detectados: {int(mask_fp_agresivo.sum())}\n"
        "Decisión: conservar `falso_positivo = True` y corregir `accion_tomada` a 'Permitido', "
        "porque un falso positivo no debería terminar en contención agresiva."
    ),
    "Lógica 5.1",
)

display(fp_agresivo_rows.head(10))

df.loc[mask_fp_agresivo, "accion_tomada"] = "Permitido"
utils.styled_output(
    f"Casos corregidos en accion_tomada: {int(mask_fp_agresivo.sum())}",
    "Resultado 5.1",
)

# Lógica 5.1
Casos detectados: 68
Decisión: conservar `falso_positivo = True` y corregir `accion_tomada` a 'Permitido', porque un falso positivo no debería terminar en contención agresiva.


,evento_id,tipo_evento,accion_tomada,falso_positivo,severidad
341,EVT103954,Malware Detectado,Bloqueado,True,Alta
356,EVT101183,Acceso No Autorizado,Bloqueado,True,Crítica
498,EVT104756,Inyección SQL,Bloqueado,True,Media
526,EVT104842,DDoS,Bloqueado,True,Crítica
741,EVT104444,Login Exitoso,Bloqueado,True,Baja
762,EVT100259,Acceso No Autorizado,Cuarentenado,True,Crítica
804,EVT102333,XSS Detectado,Bloqueado,True,Media
1033,EVT102456,Fuerza Bruta,Bloqueado,True,Alta
1048,EVT104273,Ransomware Detectado,Bloqueado,True,Crítica
1134,EVT102518,Movimiento Lateral,Bloqueado,True,Crítica


# Resultado 5.1
Casos corregidos en accion_tomada: 68


### 5.2 Resuelto sin timestamp
Detecta casos con `resuelto = True` y `timestamp_resolucion` nulo.

**Apartado de código:** usar la celda siguiente.

In [20]:
mask_resuelto_sin_timestamp = utils.resolved_without_timestamp_mask(df)
resuelto_sin_timestamp_rows = df.loc[
    mask_resuelto_sin_timestamp,
    ["evento_id", "timestamp_evento", "timestamp_resolucion", "resuelto", "analista_id"],
] .copy()

utils.styled_output(
    (
        f"Casos detectados: {int(mask_resuelto_sin_timestamp.sum())}\n"
        "Decisión: cambiar `resuelto` a False, porque sin fecha de cierre no hay evidencia suficiente "
        "para afirmar que el evento fue resuelto."
    ),
    "Lógica 5.2",
)

display(resuelto_sin_timestamp_rows.head(10))

df.loc[mask_resuelto_sin_timestamp, "resuelto"] = False
utils.styled_output(
    f"Casos reclasificados como no resueltos: {int(mask_resuelto_sin_timestamp.sum())}",
    "Resultado 5.2",
)

# Lógica 5.2
Casos detectados: 40
Decisión: cambiar `resuelto` a False, porque sin fecha de cierre no hay evidencia suficiente para afirmar que el evento fue resuelto.


,evento_id,timestamp_evento,timestamp_resolucion,resuelto,analista_id
2,EVT104669,2024-10-30 05:31:03,NaT,True,ANA010
145,EVT102018,2024-07-24 10:45:36,NaT,True,ANA002
365,EVT102765,2024-04-02 16:20:34,NaT,True,ANA012
400,EVT102687,2024-04-12 19:09:48,NaT,True,ANA006
572,EVT104686,2024-05-29 21:41:29,NaT,True,ANA004
610,EVT104930,2024-04-14 02:59:47,NaT,True,ANA010
711,EVT101158,2024-09-06 04:02:55,NaT,True,ANA014
715,EVT103492,2024-04-13 05:26:48,NaT,True,ANA011
1171,EVT100051,2024-11-16 02:19:40,NaT,True,ANA013
1347,EVT105158,2024-12-24 20:57:57,NaT,True,ANA014


# Resultado 5.2
Casos reclasificados como no resueltos: 40


### 5.3 Resolución antes del evento
Detecta y elimina filas donde `timestamp_resolucion < timestamp_evento`.

**Apartado de código:** usar la celda siguiente.

In [21]:
mask_resolucion_antes_evento = utils.resolution_before_event_mask(df)
resolucion_antes_evento_rows = df.loc[
    mask_resolucion_antes_evento,
    [
        "evento_id",
        "timestamp_evento",
        "timestamp_resolucion",
        "tiempo_respuesta_min",
        "tipo_evento",
    ],
] .copy()

utils.styled_output(
    (
        f"Casos imposibles detectados: {int(mask_resolucion_antes_evento.sum())}\n"
        "Decisión: eliminar estas filas, porque la secuencia temporal del evento quedó corrupta y "
        "no existe una forma confiable de reconstruirla."
    ),
    "Lógica 5.3",
)

display(resolucion_antes_evento_rows.head(10))

df = df.loc[~mask_resolucion_antes_evento].copy()
utils.styled_output(
    f"Filas restantes después de eliminar resoluciones imposibles: {len(df)}",
    "Resultado 5.3",
)

# Lógica 5.3
Casos imposibles detectados: 45
Decisión: eliminar estas filas, porque la secuencia temporal del evento quedó corrupta y no existe una forma confiable de reconstruirla.


,evento_id,timestamp_evento,timestamp_resolucion,tiempo_respuesta_min,tipo_evento
1,EVT104241,2024-03-16 13:18:00,2024-03-16 11:53:00,50,Exfiltración De Datos
213,EVT102436,2024-10-05 19:13:59,2024-05-10 19:54:59,41,Exfiltración De Datos
702,EVT105351,2024-12-24 16:25:23,2024-12-24 15:16:23,71,Escaneo De Red
1043,EVT100731,2024-05-20 20:08:37,2024-05-20 18:43:37,745,Login Exitoso
1211,EVT103226,2024-06-16 12:30:25,2024-06-16 10:35:25,241,Inyección SQL
1435,EVT104660,2024-11-02 12:16:44,2024-11-02 11:20:44,305,Fuerza Bruta
1564,EVT101907,2024-08-09 21:36:14,2024-08-09 19:50:14,72,Movimiento Lateral
1594,EVT100461,2024-09-05 08:08:00,2024-09-05 07:45:00,52,Exfiltración De Datos
1656,EVT102700,2024-02-12 02:49:25,2024-02-12 01:25:25,147,Phishing Detectado
1717,EVT100869,2024-07-05 00:02:27,2024-07-04 22:10:27,492,Login Exitoso


# Resultado 5.3
Filas restantes después de eliminar resoluciones imposibles: 5415


### 5.4 Tiempo de respuesta inconsistente
Crea `tiempo_calculado` desde timestamps y compara con `tiempo_respuesta_min`. Cuenta diferencias mayores a 30 minutos.

**Apartado de código:** usar la celda siguiente.

In [22]:
df["tiempo_calculado"] = utils.calculate_response_time_minutes(df)
mask_tiempo_inconsistente = utils.response_time_inconsistency_mask(
    df["tiempo_respuesta_min"],
    df["tiempo_calculado"],
    tolerance_minutes=30,
 )

tiempos_inconsistentes_rows = df.loc[
    mask_tiempo_inconsistente,
    [
        "evento_id",
        "timestamp_evento",
        "timestamp_resolucion",
        "tiempo_respuesta_min",
        "tiempo_calculado",
        "severidad",
    ],
] .copy()

utils.styled_output(
    (
        f"Registros con diferencia mayor a 30 minutos: {int(mask_tiempo_inconsistente.sum())}\n"
        "Decisión: recalcular `tiempo_respuesta_min` a partir de los timestamps, porque esas marcas "
        "son la fuente primaria del proceso."
    ),
    "Lógica 5.4",
)

display(tiempos_inconsistentes_rows.head(10))

df.loc[mask_tiempo_inconsistente, "tiempo_respuesta_min"] = df.loc[
    mask_tiempo_inconsistente, "tiempo_calculado"
].astype("Int64")

utils.styled_output(
    f"Casos corregidos en tiempo_respuesta_min: {int(mask_tiempo_inconsistente.sum())}",
    "Resultado 5.4",
)

df = df.drop(columns=["tiempo_calculado"])

# Lógica 5.4
Registros con diferencia mayor a 30 minutos: 126
Decisión: recalcular `tiempo_respuesta_min` a partir de los timestamps, porque esas marcas son la fuente primaria del proceso.


,evento_id,timestamp_evento,timestamp_resolucion,tiempo_respuesta_min,tiempo_calculado,severidad
13,EVT104224,2024-01-13 11:55:13,2024-01-13 12:35:13,2738,40,Crítica
14,EVT104867,2024-09-09 14:42:34,2024-09-09 16:03:34,4623,81,Alta
15,EVT101919,2024-08-06 09:11:59,2024-08-06 16:52:59,6119,461,Baja
55,EVT100079,2024-03-05 18:33:20,2024-05-03 21:19:20,166,85126,Alta
95,EVT100346,2024-02-24 20:26:04,2024-02-25 02:34:04,3102,368,Baja
115,EVT104692,2024-04-12 18:27:15,2024-04-12 20:33:15,5309,126,Media
129,EVT101129,2024-10-17 09:02:00,2024-10-17 09:54:00,1448,52,Crítica
137,EVT101840,2024-10-25 10:54:46,2024-10-25 13:06:46,7292,132,Alta
148,EVT102860,2024-09-25 15:00:03,2024-09-25 16:31:03,8366,91,Media
184,EVT102531,2024-01-09 20:59:10,2024-01-09 21:10:10,4149,11,Crítica


# Resultado 5.4
Casos corregidos en tiempo_respuesta_min: 126


### 5.5 Violación de SLA
Aplica reglas SLA por severidad. Reporta porcentaje de Críticos > 60 min y Altos > 240 min.

**Apartado de código:** usar la celda siguiente.

In [23]:
sla_limits = utils.sla_limit_series(df["severidad"])
mask_sla = utils.sla_violation_mask(df["tiempo_respuesta_min"], sla_limits)

sla_summary = (
    df.loc[mask_sla, "severidad"]
    .value_counts()
    .rename("casos")
    .to_frame()
)

total_violaciones_sla = int(mask_sla.sum())
criticos_con_tiempo = df["severidad"].eq("Crítica") & df["tiempo_respuesta_min"].notna()
altos_con_tiempo = df["severidad"].eq("Alta") & df["tiempo_respuesta_min"].notna()
pct_criticos_fuera_sla = float(mask_sla.loc[criticos_con_tiempo].mean() * 100) if criticos_con_tiempo.any() else 0.0
pct_altos_fuera_sla = float(mask_sla.loc[altos_con_tiempo].mean() * 100) if altos_con_tiempo.any() else 0.0

display(sla_summary)
utils.styled_output(
    (
        f"Violaciones SLA detectadas: {total_violaciones_sla}\n"
        f"Críticos por encima de 60 min: {pct_criticos_fuera_sla:.1f}%\n"
        f"Altos por encima de 240 min: {pct_altos_fuera_sla:.1f}%"
    ),
    "Resultado 5.5",
)

,casos
severidad,
Crítica,13
Alta,12
Baja,2
Media,1


# Resultado 5.5
Violaciones SLA detectadas: 28
Críticos por encima de 60 min: 0.8%
Altos por encima de 240 min: 0.8%


### 5.6 IP origen igual a IP destino
Detecta y elimina filas donde `ip_origen == ip_destino`.

**Apartado de código:** usar la celda siguiente.

In [24]:
mask_ips_iguales = utils.same_source_destination_ip_mask(df)
ips_iguales_rows = df.loc[
    mask_ips_iguales,
    ["evento_id", "ip_origen", "ip_destino", "tipo_evento", "sistema_afectado"],
] .copy()

utils.styled_output(
    (
        f"Casos detectados: {int(mask_ips_iguales.sum())}\n"
        "Decisión: eliminar estas filas, porque en este contexto SOC un host no debería atacarse a sí mismo."
    ),
    "Lógica 5.6",
)

display(ips_iguales_rows.head(10))

df = df.loc[~mask_ips_iguales].copy()
utils.styled_output(
    f"Filas restantes después de eliminar IP origen = IP destino: {len(df)}",
    "Resultado 5.6",
)

# Lógica 5.6
Casos detectados: 23
Decisión: eliminar estas filas, porque en este contexto SOC un host no debería atacarse a sí mismo.


,evento_id,ip_origen,ip_destino,tipo_evento,sistema_afectado
188,EVT100315,10.3.36.84,10.3.36.84,Movimiento Lateral,SRV-DB-DEV
420,EVT102922,10.5.50.232,10.5.50.232,Login Exitoso,ROUTER-CORE
486,EVT102794,10.10.33.151,10.10.33.151,Exfiltración De Datos,SRV-MAIL-01
600,EVT102718,10.4.28.240,10.4.28.240,Login Fallido,SRV-DB-DEV
776,EVT101168,10.0.43.204,10.0.43.204,Malware Detectado,APP-ERP-PROD
827,EVT103941,10.10.12.174,10.10.12.174,Exfiltración De Datos,SRV-DB-DEV
1207,EVT104500,10.9.13.62,10.9.13.62,Phishing Detectado,SRV-AD-PRIMARY
1232,EVT103282,10.3.29.27,10.3.29.27,Escaneo De Red,APP-ERP-PROD
1416,EVT101174,10.1.47.254,10.1.47.254,Movimiento Lateral,SRV-VPNGATEWAY
2117,EVT100941,10.0.34.38,10.0.34.38,Phishing Detectado,ROUTER-CORE


# Resultado 5.6
Filas restantes después de eliminar IP origen = IP destino: 5392


### 5.7 Login exitoso bloqueado
Detecta casos de `tipo_evento = 'Login Exitoso'` con `accion_tomada = 'Bloqueado'`.

**Apartado de código:** usar la celda siguiente.

In [25]:
mask_login_bloqueado = utils.successful_login_blocked_mask(df)
login_bloqueado_rows = df.loc[
    mask_login_bloqueado,
    ["evento_id", "tipo_evento", "accion_tomada", "usuario", "sistema_afectado"],
] .copy()

utils.styled_output(
    (
        f"Casos detectados: {int(mask_login_bloqueado.sum())}\n"
        "Decisión: corregir `accion_tomada` a 'Permitido', porque un login exitoso es incompatible "
        "con una acción de bloqueo."
    ),
    "Lógica 5.7",
)

display(login_bloqueado_rows.head(10))

df.loc[mask_login_bloqueado, "accion_tomada"] = "Permitido"
utils.styled_output(
    f"Casos corregidos en accion_tomada: {int(mask_login_bloqueado.sum())}",
    "Resultado 5.7",
)

# Lógica 5.7
Casos detectados: 74
Decisión: corregir `accion_tomada` a 'Permitido', porque un login exitoso es incompatible con una acción de bloqueo.


,evento_id,tipo_evento,accion_tomada,usuario,sistema_afectado
41,EVT101498,Login Exitoso,Bloqueado,user_hernandez13,SRV-MONITOR
142,EVT102053,Login Exitoso,Bloqueado,user_gonzalez96,SRV-VPNGATEWAY
197,EVT103047,Login Exitoso,Bloqueado,user_perez38,SRV-AD-PRIMARY
257,EVT102784,Login Exitoso,Bloqueado,user_rodriguez95,SRV-MONITOR
374,EVT100245,Login Exitoso,Bloqueado,user_hernandez19,SRV-DB-DEV
422,EVT103011,Login Exitoso,Bloqueado,user_gonzalez13,APP-ERP-PROD
477,EVT103967,Login Exitoso,Bloqueado,user_lopez21,APP-ERP-PROD
523,EVT101350,Login Exitoso,Bloqueado,user_rodriguez39,FW-PERIMETRO
531,EVT100734,Login Exitoso,Bloqueado,user_garcia69,SRV-DB-DEV
629,EVT100724,Login Exitoso,Bloqueado,user_garcia66,SRV-MAIL-01


# Resultado 5.7
Casos corregidos en accion_tomada: 74


### 5.8 FP crítico no resuelto
Detecta registros con `falso_positivo = True`, `severidad = 'Crítica'` y `resuelto = False`. Interpreta qué indica sobre triage.

**Apartado de código:** usar la celda siguiente.

In [26]:
mask_fp_critico_no_resuelto = utils.critical_false_positive_unresolved_mask(df)
fp_critico_no_resuelto_rows = df.loc[
    mask_fp_critico_no_resuelto,
    [
        "evento_id",
        "severidad",
        "falso_positivo",
        "resuelto",
        "timestamp_resolucion",
        "analista_id",
    ],
] .copy()

mask_fp_critico_cerrable = mask_fp_critico_no_resuelto & df["timestamp_resolucion"].notna()
mask_fp_critico_revision = mask_fp_critico_no_resuelto & df["timestamp_resolucion"].isna()

utils.styled_output(
    (
        f"Casos sospechosos detectados: {int(mask_fp_critico_no_resuelto.sum())}\n"
        f"Marcados como resueltos por evidencia de cierre: {int(mask_fp_critico_cerrable.sum())}\n"
        f"Conservados para revisión manual por falta de timestamp: {int(mask_fp_critico_revision.sum())}\n"
        "Interpretación: esta combinación sugiere un proceso de triage inconsistente, donde la "
        "clasificación de falso positivo y el estado operativo no se actualizaron de forma coordinada."
    ),
    "Resultado 5.8",
)

display(fp_critico_no_resuelto_rows.head(10))

df.loc[mask_fp_critico_cerrable, "resuelto"] = True

# Resultado 5.8
Casos sospechosos detectados: 22
Marcados como resueltos por evidencia de cierre: 13
Conservados para revisión manual por falta de timestamp: 9
Interpretación: esta combinación sugiere un proceso de triage inconsistente, donde la clasificación de falso positivo y el estado operativo no se actualizaron de forma coordinada.


,evento_id,severidad,falso_positivo,resuelto,timestamp_resolucion,analista_id
123,EVT101491,Crítica,True,False,2024-03-09 07:55:02,ANA002
501,EVT100106,Crítica,True,False,NaT,ANA010
526,EVT104842,Crítica,True,False,NaT,ANA014
762,EVT100259,Crítica,True,False,2024-07-24 14:37:48,ANA002
1048,EVT104273,Crítica,True,False,2024-03-29 13:06:59,ANA002
1425,EVT105166,Crítica,True,False,2024-12-26 12:44:34,ANA006
1517,EVT103780,Crítica,True,False,2024-01-18 22:38:23,ANA005
1611,EVT104418,Crítica,True,False,NaT,ANA013
1826,EVT103071,Crítica,True,False,NaT,ANA001
1835,EVT101102,Crítica,True,False,2024-09-30 18:59:13,ANA003


## Fase 6 - Imputación y Decisiones Finales

### 6.1 Nulos en `usuario`
Decide si imputar o dejar nulos y justifica según el contexto de seguridad.

**Apartado de decisión (markdown) y código (si aplica):** usar celdas siguientes.

**Justificación 6.1:** conviene dejar los nulos de `usuario` tal como están. En un SIEM muchos eventos provienen de tráfico de red, escaneos o actividad automatizada sin una cuenta autenticada asociada. Imputar un valor artificial mezclaría ausencia real de identidad con un dato inventado y podría sesgar análisis posteriores sobre cuentas comprometidas.

In [27]:
usuario_nulos = int(df["usuario"].isna().sum())
utils.styled_output(
    (
        f"Nulos en usuario conservados: {usuario_nulos}\n"
        "Decisión aplicada: no imputar, porque la ausencia de usuario puede ser legítima en eventos de red."
    ),
    "Resultado 6.1",
)

# Resultado 6.1
Nulos en usuario conservados: 839
Decisión aplicada: no imputar, porque la ausencia de usuario puede ser legítima en eventos de red.


### 6.2 Nulos en `pais_origen`
Decide entre imputar con `Desconocido` o eliminar filas y explica impacto en el análisis.

**Apartado de decisión (markdown) y código (si aplica):** usar celdas siguientes.

**Justificación 6.2:** para `pais_origen` sí conviene imputar `Desconocido`. El evento sigue siendo útil para métricas de volumen, severidad y tiempos de respuesta, y eliminar la fila perdería señal operativa real. Usar una categoría explícita permite conservar el registro sin fingir una geolocalización concreta.

In [28]:
pais_origen_nulos_antes = int(df["pais_origen"].isna().sum())
df["pais_origen"] = df["pais_origen"].fillna("Desconocido")
pais_origen_nulos_despues = int(df["pais_origen"].isna().sum())

utils.styled_output(
    (
        f"Nulos antes de imputar: {pais_origen_nulos_antes}\n"
        f"Nulos después de imputar: {pais_origen_nulos_despues}\n"
        "Decisión aplicada: se imputó 'Desconocido' para conservar el evento sin asumir un país real."
    ),
    "Resultado 6.2",
)

# Resultado 6.2
Nulos antes de imputar: 15
Nulos después de imputar: 0
Decisión aplicada: se imputó 'Desconocido' para conservar el evento sin asumir un país real.


### 6.3 Nulos en `puerto_destino`
Evalúa si imputar (ej. mediana) tiene sentido o si debe conservarse como ausencia de dato.

**Apartado de decisión (markdown) y código (si aplica):** usar celdas siguientes.

**Justificación 6.3:** no tiene sentido imputar `puerto_destino` con la mediana ni con otro valor central. Un puerto faltante no representa un puerto intermedio típico, sino ausencia de información sobre el servicio objetivo. Inventarlo distorsionaría análisis de superficie de ataque y distribución de servicios; por eso es mejor conservar el null y excluirlo solo en análisis que requieran ese campo.

In [29]:
puerto_nulos = int(df["puerto_destino"].isna().sum())
utils.styled_output(
    (
        f"Nulos en puerto_destino conservados: {puerto_nulos}\n"
        "Decisión aplicada: no imputar, porque un puerto faltante representa ausencia de dato y no un valor típico."
    ),
    "Resultado 6.3",
)

# Resultado 6.3
Nulos en puerto_destino conservados: 12
Decisión aplicada: no imputar, porque un puerto faltante representa ausencia de dato y no un valor típico.


### 6.4 Aplicación de decisiones finales
Aplica tus decisiones y reporta cuántos nulos quedan al final.

**Apartado de código:** usar la celda siguiente.

In [30]:
analista_nulos_antes = int(df["analista_id"].isna().sum())
df["analista_id"] = df["analista_id"].fillna("SIN_ASIGNAR")
analista_nulos_despues = int(df["analista_id"].isna().sum())

remaining_nulls = utils.remaining_nulls_summary(df)

utils.styled_output(
    (
        f"Nulos en analista_id antes de imputar: {analista_nulos_antes}\n"
        f"Nulos en analista_id después de imputar: {analista_nulos_despues}\n"
        f"Columnas con nulos restantes: {remaining_nulls.shape[0]}"
    ),
    "Resultado 6.4",
)

display(remaining_nulls if not remaining_nulls.empty else pd.DataFrame({"nulos": []}))

# Resultado 6.4
Nulos en analista_id antes de imputar: 19
Nulos en analista_id después de imputar: 0
Columnas con nulos restantes: 4


,nulos
timestamp_resolucion,1001
usuario,839
puerto_destino,12
sistema_operativo,9


## Fase 7 - Análisis y Visualización

### 7.1 Tasa de falsos positivos por tipo de evento
Calcula la métrica `tasa_fp` por `tipo_evento`.

**Apartado de código:** usar la celda siguiente.

### 7.2 Gráfica de tasa de FP
Crea una barra horizontal ordenada, con línea vertical punteada del promedio global.

**Apartado de código (visualización 1):** usar la celda siguiente.

### 7.3 Críticos sin resolver por sistema operativo
Filtra `severidad = 'Crítica'` y `resuelto = False`, agrupa por `sistema_operativo` y cuenta.

**Apartado de código:** usar la celda siguiente.

### 7.4 Segunda gráfica por sistema operativo
Genera una segunda gráfica (barras o treemap) para los SO con más críticos sin resolver (usa rojo o escala de calor).

**Apartado de código (visualización 2):** usar la celda siguiente.

### 7.5 Conclusión final para el SOC
Responde ambas preguntas del SOC con números concretos e incluye cuántos eventos violaron el SLA.

**Apartado de conclusión:** completar esta celda markdown.

## Criterios de Evaluación (Referencia)
- Diagnóstico documentado: 10
- Limpieza de texto: 15
- Timestamps y tipos: 10
- Valores imposibles: 10
- Inconsistencias lógicas: 30
- Imputación de nulos: 10
- Visualizaciones: 10
- Conclusión del SOC: 5
- Total: 100